# License Migrator

## Setup Source & Library

### Setup Source File and Target Directory

In [1]:
source = './source/mapping-lisensi-user.csv'
converted_dir = 'converted/'
converted_file_name = 'converted_'+source.split('/')[-1]

### Setup Pandas

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(source)

## Mappings

### Mapping Windows 10 Pro / 11 Pro

In [3]:
def filter_win_license():
    filter_out_keyword = ['macbook','sudah pro', '']
    tmp = ~(df['WIN10/11PRO'].str.lower().str.strip().isin(filter_out_keyword) | df['WIN10/11PRO'].isna())
    return df[tmp][['TAG/SN','WIN10/11PRO']]

win_key_pattern = r'(\w{5}-\w{5}-\w{5}-\w{5}-\w{5})'

win_df = filter_win_license()

win_df['WIN10/11PRO'] = win_df['WIN10/11PRO'].str.replace(r' +', '', regex=True)
win_df['Notes'] = np.where(
    (win_df['WIN10/11PRO'].str.len() == 29),
    '',
    win_df['WIN10/11PRO']
)
win_df['Serial/Product Key'] = win_df['WIN10/11PRO'].str.extract(win_key_pattern, expand=False)

win_df.drop(columns=['WIN10/11PRO'], inplace=True)

win_df.rename(columns={
    'TAG/SN': 'Checked Out to: Asset Tag'
}, inplace=True)

win_df['Checked Out to: Asset Tag'] = win_df['Checked Out to: Asset Tag'].str.strip()

win_df['License Name'] = 'Windows 10/11 Pro'
win_df['Manufacturer'] = 'Microsoft'
win_df['Seats'] = 1
win_df['Min. QTY'] = 0
win_df['Category'] = 'Misc Software'

# win_df.to_csv(converted_dir+'win_'+converted_file_name, index=False)

### Mapping MS Office

In [4]:
def filter_office_license():
    filter_out_keyword = ['']
    tmp = ~(df['MICROSOFT OFFICE'].str.lower().str.strip().isin(filter_out_keyword) | df['MICROSOFT OFFICE'].isna())
    return df[tmp][['TAG/SN','MICROSOFT OFFICE', 'NAMA']]

office_key_pattern = r'(\w{5}-\w{5}-\w{5}-\w{5}-\w{5})'
office_email_pattern = r'(\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b)'

office_df = filter_office_license()

office_df['tmp_office_key'] = office_df['MICROSOFT OFFICE'].str.extract(office_key_pattern, expand=False)
office_df['tmp_office_email'] = office_df['MICROSOFT OFFICE'].str.extract(office_email_pattern, expand=False)

office_df['Notes'] = np.where(
    ((office_df['MICROSOFT OFFICE'].str.len() == office_df['tmp_office_key'].str.len()) | (office_df['MICROSOFT OFFICE'].str.len() == office_df['tmp_office_email'].str.len())),
    pd.NA,
    office_df['MICROSOFT OFFICE']
)

office_df['License Name'] = np.where(office_df['tmp_office_key'].notna(), 'Microsoft Office', 'Microsoft Office 365')
office_df['Checked Out to: Asset Tag'] = office_df['TAG/SN'].str.strip()
office_df['Serial/Product Key'] = np.where(office_df['tmp_office_key'].notna(), office_df['tmp_office_key'], office_df['tmp_office_email'])

office_df['Manufacturer'] = 'Microsoft'
office_df['Seats'] = 1
office_df['Min. QTY'] = 0
office_df['Category'] = 'Misc Software'

office_df.drop(columns=[
    'TAG/SN',
    'MICROSOFT OFFICE',
    'NAMA',
    'tmp_office_key',
    'tmp_office_email',
], inplace=True)


### Mapping Microsoft Project

In [5]:
def filter_ms_project_license():
    filter_out_keyword = ['']
    tmp = ~(df['M.PROJECT'].str.lower().str.strip().isin(filter_out_keyword) | df['M.PROJECT'].isna())
    return df[tmp][['M.PROJECT','TAG/SN']]

ms_project_df = filter_ms_project_license()

ms_project_key_pattern = r'(\w{5}-\w{5}-\w{5}-\w{5}-\w{5})'

ms_project_df['Checked Out to: Asset Tag'] = ms_project_df['TAG/SN'].str.strip()
ms_project_df['Serial/Product Key'] = ms_project_df['M.PROJECT'].str.extract(ms_project_key_pattern, expand=False)
ms_project_df['Notes'] = np.where(
    ((ms_project_df['M.PROJECT'].str.len() == ms_project_df['Serial/Product Key'].str.len())),
    pd.NA,
    ms_project_df['M.PROJECT']
)

ms_project_df['License Name'] = 'Microsoft Project'
ms_project_df['Manufacturer'] = 'Microsoft'
ms_project_df['Seats'] = 1
ms_project_df['Min. QTY'] = 0
ms_project_df['Category'] = 'Misc Software'

ms_project_df.drop(columns=[
    'TAG/SN',
    'M.PROJECT',
], inplace=True)

### Mapping Microsoft Visio

In [6]:
def filter_visio_license():
    filter_out_keyword = ['']
    tmp = ~(df['Visio'].str.lower().str.strip().isin(filter_out_keyword) | df['Visio'].isna())
    return df[tmp][['Visio','TAG/SN']]

visio_df = filter_visio_license()

visio_key_pattern = r'(\w{5}-\w{5}-\w{5}-\w{5}-\w{5})'

visio_df['Checked Out to: Asset Tag'] = visio_df['TAG/SN'].str.strip()
visio_df['Serial/Product Key'] = visio_df['Visio'].str.extract(visio_key_pattern, expand=False)
visio_df['Notes'] = np.where(
    ((visio_df['Visio'].str.len() == visio_df['Serial/Product Key'].str.len())),
    pd.NA,
    visio_df['Visio']
)

visio_df['License Name'] = 'Microsoft Visio'
visio_df['Manufacturer'] = 'Microsoft'
visio_df['Seats'] = 1
visio_df['Min. QTY'] = 0
visio_df['Category'] = 'Misc Software'

visio_df.drop(columns=[
    'TAG/SN',
    'Visio',
], inplace=True)

## Merge and Export to converted CSV

In [7]:

merged_df = pd.concat([win_df, office_df, ms_project_df, visio_df], ignore_index=True)
merged_df.to_csv(converted_dir+converted_file_name, index=False)